# FLSTM Gates Only using SG
Dataset: Mackey-Glass — Lag=90

**Classification Metrics Evaluation (Mode Threshold, No Noise)**

In [ ]:
# ============================================================
# PROCESS IDENTIFICATION
# ============================================================
import os
print(f"Process ID (PID): {os.getpid()}")


In [ ]:
# ============================================================
# NOTEBOOK TIMER — START
# ============================================================
import time as _timer_module
_NOTEBOOK_START_TIME = _timer_module.time()
print(f"Notebook execution started at: {_timer_module.strftime('%Y-%m-%d %H:%M:%S')}")


In [ ]:
import os
import glob
import warnings
import gc
import numpy as np
import pandas as pd
from math import sqrt
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, roc_auc_score

# Suppress expected UserWarnings
warnings.filterwarnings('ignore', category=UserWarning, module='keras')

# ------------------------------------------------------------
# 1. DATA LOADING & PREPROCESSING
# ------------------------------------------------------------
import openpyxl
file_path = next((p for p in [
    '../content/Mackey-Glass Time Series(taw17).xlsx',
    'content/Mackey-Glass Time Series(taw17).xlsx',
    '/Users/satabarto/Research/content/Mackey-Glass Time Series(taw17).xlsx',
    '/kaggle/input/datasets/saurabhshahane/mackey-glass-time-series/Mackey-Glass Time Series(taw17).xlsx'
] if os.path.exists(p)), '../content/Mackey-Glass Time Series(taw17).xlsx')
series = pd.read_excel(file_path)
raw_values = series['t+1'].values.flatten()
original_raw_values = np.copy(raw_values)

# ------------------------------------------------------------
# 2. TIME SERIES & LAG SETUP
# ------------------------------------------------------------
LAG_STEPS = 90  # Configurable lag step parameter

def difference(dataset, interval=1):
    diff = []
    for i in range(interval, len(dataset)):
        diff.append(dataset[i] - dataset[i - interval])
    return np.array(diff)

def timeseries_to_supervised(data, lag):
    df = pd.DataFrame(data)
    columns = [df.shift(i) for i in range(1, lag+1)]
    columns.append(df)
    df = pd.concat(columns, axis=1)
    df.fillna(0, inplace=True)
    return df.values

diff_values = difference(raw_values, 1)
supervised = timeseries_to_supervised(diff_values, LAG_STEPS)

train, test = supervised[:-60], supervised[-60:]
scaler = MinMaxScaler(feature_range=(-1, 1))
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)

X_train_raw, y_train = train_scaled[:, 0:-1], train_scaled[:, -1]
X_test_raw, y_test = test_scaled[:, 0:-1], test_scaled[:, -1]

# ------------------------------------------------------------
# 3. WANG-MENDEL FUZZY RULE BASE (No complement rules)
# ------------------------------------------------------------
Q_REGIONS = 5

def triangular_mf_np(x, a, b, c):
    return np.maximum(0.0, np.minimum((x - a) / (b - a + 1e-10), (c - x) / (c - b + 1e-10)))

def build_triangular_fuzzy_sets(data_min, data_max, q=Q_REGIONS):
    centers = np.linspace(data_min, data_max, q)
    step = centers[1] - centers[0] if q > 1 else 1.0
    fuzzy_sets = [(center - step, center, center + step) for center in centers]
    return fuzzy_sets, centers

def fuzzify_value(value, fuzzy_sets):
    best_idx, best_mu = 0, 0.0
    for i, (a, b, c) in enumerate(fuzzy_sets):
        mu = triangular_mf_np(value, a, b, c)
        if mu > best_mu:
            best_mu = mu
            best_idx = i
    return best_idx, max(best_mu, 1e-10)

def extract_wm_rules(X, y, fs_x_list, fs_y):
    rules = {}
    n_features = X.shape[1]
    for i in range(len(X)):
        antecedents = []
        weight = 1.0
        for j in range(n_features):
            idx, mu = fuzzify_value(X[i, j], fs_x_list[j])
            antecedents.append(idx)
            weight *= mu
        cons_idx, cons_mu = fuzzify_value(y[i], fs_y)
        weight *= cons_mu
        key = tuple(antecedents)
        if key not in rules or weight > rules[key][1]:
            rules[key] = (cons_idx, weight)
    return rules

def wm_fuzzy_predict(x_features, rules, fs_x_list, centers_y, q=Q_REGIONS):
    n_features = len(x_features)
    memberships = []
    for j in range(n_features):
        mf_vals = [triangular_mf_np(x_features[j], fs_x_list[j][k][0], fs_x_list[j][k][1], fs_x_list[j][k][2]) for k in range(q)]
        memberships.append(mf_vals)
    
    total_weight = 0.0
    weighted_sum = 0.0
    for antecedents, (consequent, _) in rules.items():
        strength = 1.0
        for j, ant_idx in enumerate(antecedents):
            strength *= memberships[j][ant_idx]
        if strength > 1e-10:
            weighted_sum += strength * centers_y[consequent]
            total_weight += strength
            
    if total_weight > 1e-10:
        return weighted_sum / total_weight
    # Fallback default value if no rules trigger
    return centers_y[q // 2]

def build_wm_system(X_train, y_train, q=Q_REGIONS):
    n_features = X_train.shape[1]
    fs_x_list = []
    for j in range(n_features):
        col = X_train[:, j]
        fs, _ = build_triangular_fuzzy_sets(col.min(), col.max(), q)
        fs_x_list.append(fs)
    fs_y, centers_y = build_triangular_fuzzy_sets(y_train.min(), y_train.max(), q)
    
    # Extract rules without creating full combinatorial grid
    rules = extract_wm_rules(X_train, y_train, fs_x_list, fs_y)
    return rules, fs_x_list, centers_y

def compute_fuzzy_predictions(X, rules, fs_x_list, centers_y, q=Q_REGIONS):
    preds = np.zeros(len(X))
    for i in range(len(X)):
        preds[i] = wm_fuzzy_predict(X[i], rules, fs_x_list, centers_y, q)
    return preds

# Build WM fuzzy system from dynamic-lag training data
wm_rules, wm_fs_x, wm_centers_y = build_wm_system(X_train_raw, y_train)

# Compute fuzzy predictions r_t
r_train = compute_fuzzy_predictions(X_train_raw, wm_rules, wm_fs_x, wm_centers_y)
r_test = compute_fuzzy_predictions(X_test_raw, wm_rules, wm_fs_x, wm_centers_y)

# Concatenate r_t as last feature: [x_t, r_t]
X_train_aug = np.column_stack([X_train_raw, r_train])
X_test_aug = np.column_stack([X_test_raw, r_test])

num_features = X_train_aug.shape[1]  # Equal to LAG_STEPS + 1

X_train = X_train_aug.reshape((X_train_aug.shape[0], 1, num_features))
X_test = X_test_aug.reshape((X_test_aug.shape[0], 1, num_features))

# ------------------------------------------------------------
# 4. FLSTM CELL WITH SG & MEMORY LAYER
# ------------------------------------------------------------
def signed_gaussian(x, sigma=0.5):
    """Signed Gaussian (SG) Membership Function"""
    return x * tf.exp(-tf.square(x) / (2 * sigma**2))

@tf.keras.utils.register_keras_serializable()
class MembershipFLSTMSNPCell(layers.Layer):
    def __init__(self, units, mf_type='sg', **kwargs):
        super().__init__(**kwargs)
        self.units, self.mf_type = units, mf_type
        self.state_size, self.output_size = (units, units, units), units
        self.mf = signed_gaussian

    def build(self, input_shape):
        total_input_dim = input_shape[-1]
        actual_input_dim = total_input_dim - 1  # Exclude r_t
        self.kernel = self.add_weight(shape=(actual_input_dim, self.units * 4), initializer='glorot_uniform', name='kernel')
        self.recurrent_kernel = self.add_weight(shape=(self.units, self.units * 4), initializer='orthogonal', name='recurrent_kernel')
        self.bias = self.add_weight(shape=(self.units * 4,), initializer='zeros', name='bias')
        self.fuzzy_gate_kernel = self.add_weight(shape=(1, self.units * 3), initializer='glorot_uniform', name='fuzzy_gate_kernel')

    def call(self, inputs, states):
        u_tm1 = states[0]
        x_t = inputs[:, :-1]
        r_t = inputs[:, -1:]
        
        z = tf.matmul(x_t, self.kernel) + tf.matmul(u_tm1, self.recurrent_kernel) + self.bias
        z0, z1, z2, z3 = z[:, :self.units], z[:, self.units:2*self.units], z[:, 2*self.units:3*self.units], z[:, 3*self.units:]
        
        fz = tf.matmul(r_t, self.fuzzy_gate_kernel)
        z0 = z0 + fz[:, :self.units]
        z1 = z1 + fz[:, self.units:2*self.units]
        z2 = z2 + fz[:, 2*self.units:]

        r = tf.tanh(z0)
        c = tf.clip_by_value(self.mf(z1), -1.0, 1.0)
        o = tf.clip_by_value(self.mf(z2), -1.0, 1.0)
        a = tf.tanh(z3)

        u = r * u_tm1 - c * a
        h = o * a
        return h, [u, c, o]

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units, 'mf_type': self.mf_type})
        return config

@tf.keras.utils.register_keras_serializable()
class StrengtheningMemoryLayer(layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units_out = units
        self.dense = layers.Dense(units, activation='tanh')

    def call(self, inputs):
        h_t, c_t = inputs
        ch_t = h_t + c_t
        s_t = self.dense(ch_t)
        h_hat = ch_t + s_t
        return h_hat

    def get_config(self):
        config = super().get_config()
        config.update({'units': self.units_out})
        return config

def build_model(input_dim, units=8, batch_size=1):
    cell = MembershipFLSTMSNPCell(units, mf_type='sg')
    rnn = layers.RNN(cell, return_sequences=False, return_state=True, stateful=True)

    inputs = tf.keras.Input(batch_shape=(batch_size, 1, input_dim))
    x, u_out, c_out, o_out = rnn(inputs)
    h_strengthened = StrengtheningMemoryLayer(units)([x, u_out])
    outputs = layers.Dense(1)(h_strengthened)

    model = tf.keras.Model(inputs=inputs, outputs=[outputs, c_out, o_out])
    model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss=['mean_squared_error', None, None])
    return model

# ------------------------------------------------------------
# 5. TRAINING & EVALUATION (2 Runs)
# ------------------------------------------------------------
print('\n' + '='*80)
print(f'EVALUATING: Gates Only (SG) on Mackey-Glass — Lag Steps: {LAG_STEPS}')
print('='*80 + '\n')

all_rmse, all_mae, all_predictions, all_losses = [], [], [], []
all_accuracy, all_specificity, all_precision, all_recall, all_auc = [], [], [], [], []

for run in range(2):
    print(f'\\n===== RUN {run+1}/2 =====')
    np.random.seed(run)
    tf.random.set_seed(run)
    
    tf.keras.backend.clear_session()
    gc.collect()

    model = build_model(input_dim=num_features, units=8, batch_size=1)
    rnn_layer = model.layers[1]

    run_losses = []
    for epoch in range(100):
        history = model.fit(X_train, y_train, epochs=1, batch_size=1, verbose=0, shuffle=False)
        run_losses.append(history.history['loss'][0])
        rnn_layer.reset_states()
    all_losses.append(run_losses)

    # Warmup RNN states (Direct model call avoids memory build-up)
    for i in range(len(X_train)): 
        _ = model(X_train[i:i+1], training=False)

    predictions = []
    for i in range(len(X_test)):
        # Direct functional call instead of model.predict inside loops
        yhat, c_val, o_val = model(X_test[i:i+1], training=False)

        row = list(X_test_raw[i]) + [yhat.numpy()[0, 0]]
        diff_pred = scaler.inverse_transform([row])[0, -1]
        
        prev_actual_idx = len(train) + i
        inv = diff_pred + raw_values[prev_actual_idx]
        predictions.append(inv)

    actual = raw_values[-60:]
    rmse = sqrt(mean_squared_error(actual, predictions))
    mae = np.mean(np.abs(np.array(actual) - np.array(predictions)))

    all_rmse.append(rmse)
    all_mae.append(mae)
    all_predictions.append(predictions)

    # ---- Classification Metrics (Dynamic Threshold: Mode) ----
    import scipy.stats as stats
    train_raw_values = raw_values[:-60]
    train_diffs = np.diff(train_raw_values)
    rounded_diffs = np.round(train_diffs, 3)
    mode_res = stats.mode(rounded_diffs, keepdims=True)
    tau = float(mode_res.mode[0]) if hasattr(mode_res.mode, '__len__') else float(mode_res.mode)
    y_prev = raw_values[-(len(actual)+1):-1]
    true_dir = ((np.array(actual) - y_prev) > tau).astype(int)
    pred_dir = ((np.array(predictions) - y_prev) > tau).astype(int)
    pred_scores = (np.array(predictions) - y_prev) - tau

    TP = np.sum((true_dir == 1) & (pred_dir == 1))
    TN = np.sum((true_dir == 0) & (pred_dir == 0))
    FP = np.sum((true_dir == 0) & (pred_dir == 1))
    FN = np.sum((true_dir == 1) & (pred_dir == 0))

    accuracy = (TP + TN) / len(true_dir) if len(true_dir) > 0 else 0.0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    try:
        auc = roc_auc_score(true_dir, pred_scores)
    except ValueError:
        auc = 0.5

    all_accuracy.append(accuracy)
    all_specificity.append(specificity)
    all_precision.append(precision)
    all_recall.append(recall)
    all_auc.append(auc)

    print(f'Run {run+1} \u2014 Acc: {accuracy*100:.1f}%, Spec: {specificity*100:.1f}%, Prec: {precision*100:.1f}%, Recall: {recall*100:.1f}%, AUC: {auc*100:.1f}%')

# ------------------------------------------------------------
# 6. SUMMARY & PLOTS
# ------------------------------------------------------------
mean_accuracy = np.mean(all_accuracy) * 100
mean_specificity = np.mean(all_specificity) * 100
mean_precision = np.mean(all_precision) * 100
mean_recall = np.mean(all_recall) * 100
mean_auc = np.mean(all_auc) * 100

print(f'\\n===== FINAL RESULTS \u2014 Gates Only (SG) on Mackey-Glass (2 runs, Lag={LAG_STEPS}) =====')
print(f'Accuracy:    {mean_accuracy:.1f}%')
print(f'Specificity: {mean_specificity:.1f}%')
print(f'Precision:   {mean_precision:.1f}%')
print(f'Recall:      {mean_recall:.1f}%')
print(f'AUC:         {mean_auc:.1f}%')

best_idx = np.argmin(all_rmse)
actual = raw_values[-60:]
best_predictions = all_predictions[best_idx]

plt.figure(figsize=(12, 5))
plt.plot(actual, label='Actual', color='blue', linewidth=1.5)
plt.plot(best_predictions, label='Predicted (Best Run)', color='purple', linewidth=1.5, linestyle='--')
plt.title(f'Gates Only (SG) \u2014 Mackey-Glass (Lag={LAG_STEPS})\\nPredictions vs Actual (Best Run)')
plt.xlabel('Time Step')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# COMBINED CLASSIFICATION SUMMARY TABLE
# ============================================================
print('\n' + '\u2554' + '\u2550'*85 + '\u2557')
print('\u2551 CLASSIFICATION RESULTS \u2014 Gates Only (SG) on Mackey-Glass (Lag={LAG_STEPS}) '.ljust(86) + '\u2551')
print('\u2560' + '\u2550'*15 + '\u2566' + '\u2550'*15 + '\u2566' + '\u2550'*15 + '\u2566' + '\u2550'*15 + '\u2566' + '\u2550'*18 + '\u2563')
print('\u2551 Accuracy      \u2551 Specificity   \u2551 Precision     \u2551 Recall        \u2551 AUC              \u2551')
print('\u2560' + '\u2550'*15 + '\u256c' + '\u2550'*15 + '\u256c' + '\u2550'*15 + '\u256c' + '\u2550'*15 + '\u256c' + '\u2550'*18 + '\u2563')
acc_str = f'{mean_accuracy:.1f}%'
spec_str = f'{mean_specificity:.1f}%'
prec_str = f'{mean_precision:.1f}%'
rec_str = f'{mean_recall:.1f}%'
auc_str = f'{mean_auc:.1f}%'
print(f'\u2551 {acc_str:<13} \u2551 {spec_str:<13} \u2551 {prec_str:<13} \u2551 {rec_str:<13} \u2551 {auc_str:<16} \u2551')
print('\u255a' + '\u2550'*15 + '\u2569' + '\u2550'*15 + '\u2569' + '\u2550'*15 + '\u2569' + '\u2550'*15 + '\u2569' + '\u2550'*18 + '\u255d')


## Observations

### Gates Only (SG) on Mackey-Glass — Lag=90

**Run the notebook to generate results.**


In [ ]:
# ============================================================
# NOTEBOOK TIMER — END
# ============================================================
import time as _timer_module
_NOTEBOOK_END_TIME = _timer_module.time()
_NOTEBOOK_ELAPSED = _NOTEBOOK_END_TIME - _NOTEBOOK_START_TIME
_hours, _rem = divmod(_NOTEBOOK_ELAPSED, 3600)
_minutes, _seconds = divmod(_rem, 60)
print(f"\nTotal notebook execution time: {int(_hours)}h {int(_minutes)}m {_seconds:.2f}s")
print(f"Total seconds: {_NOTEBOOK_ELAPSED:.2f}")
